# 面试题：Embedding 是什么，如何从真实共现数据手写训练并验证它学到了语义？

## 面试回答主线

Embedding 是一个可学习查表矩阵：离散 token id 选择其中一行，得到可参与梯度计算的稠密向量。它的“语义”不是天然存在，而是由训练目标和数据分布塑造；Skip-gram negative sampling 会拉近真实上下文 pair、推远噪声 pair。评估不能只看 loss，还要在固定查询上比较近邻、相关 pair 与无关 pair 的余弦相似度，并检查高频词偏置。句向量池化时必须用 attention mask 排除 PAD，否则同一句话换一个 batch 最大长度就会改变向量。线上还必须版本化词表、embedding 权重和 pooling 规则。

## 真实案例：电子配件浏览会话

每条记录表示匿名会话中连续浏览的商品属性，`count` 是该模式在统计窗口中的出现次数。样本是结构真实的教学数据，不含用户标识；它只验证训练机制，不能代表线上召回收益。

In [1]:
from collections import Counter  # 导入计数器以聚合上下文 pair 与 token 频次。
import torch  # 导入 PyTorch 以手写可反向传播的 embedding 模型。
from torch import nn  # 导入最基础的模块和参数抽象。
import torch.nn.functional as F  # 导入 softplus 以稳定实现负采样损失。
torch.set_num_threads(1)  # 小张量教学实验固定单线程以避免线程调度开销并保证快速复现。
sessions = [  # 构造带业务频次的匿名商品浏览会话。
    (["无线", "蓝牙", "耳机", "降噪"], 120),  # 无线耳机场景强化同义属性与核心品类共现。
    (["蓝牙", "耳机", "通话", "降噪"], 100),  # 通话耳机场景继续连接蓝牙和降噪。
    (["无线", "运动", "耳机", "佩戴"], 80),  # 运动场景提供无线耳机的另一类上下文。
    (["有线", "耳机", "麦克风", "电竞"], 70),  # 有线电竞耳机形成可区分的局部语义。
    (["蓝牙", "音箱", "低音", "便携"], 75),  # 蓝牙音箱让蓝牙跨品类出现。
    (["无线", "音箱", "户外", "便携"], 60),  # 户外音箱连接无线与便携属性。
    (["快充", "充电器", "typec", "手机"], 90),  # 充电场景形成与音频不同的语义簇。
    (["手机", "保护壳", "防摔", "透明"], 65),  # 手机保护场景提供配件关系。
    (["平板", "保护壳", "支架", "防摔"], 50),  # 平板场景复用保护壳和防摔属性。
]  # 结束匿名会话列表。
print("会话样本                         出现次数")  # 输出真实案例输入表标题。
for tokens, count in sessions:  # 逐条展示会话 token 和聚合频次。
    print(f"{' → '.join(tokens):<32} {count:>6}")  # 输出有业务语义的浏览顺序。
print("聚合会话总数：", sum(count for _, count in sessions))  # 输出统计权重对应的总体规模。

会话样本                         出现次数
无线 → 蓝牙 → 耳机 → 降噪                   120
蓝牙 → 耳机 → 通话 → 降噪                   100
无线 → 运动 → 耳机 → 佩戴                    80
有线 → 耳机 → 麦克风 → 电竞                   70
蓝牙 → 音箱 → 低音 → 便携                    75
无线 → 音箱 → 户外 → 便携                    60
快充 → 充电器 → typec → 手机                90
手机 → 保护壳 → 防摔 → 透明                   65
平板 → 保护壳 → 支架 → 防摔                   50
聚合会话总数： 710


## Baseline：one-hot 向量只表达“是否同一个 token”

one-hot 的不同 token 两两正交，因此“无线”和“蓝牙”即使在相似上下文中频繁共现，相似度仍为零。先把这个最简单基线与三组业务 pair 明确打印出来。

In [2]:
all_words = sorted({token for tokens, _ in sessions for token in tokens})  # 收集训练数据实际出现的普通 token。
vocabulary = ["[PAD]"] + all_words  # 把 padding 特殊 token 固定在编号零。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 创建稳定的 token 到整数 id 映射。
one_hot = torch.eye(len(vocabulary))  # 构造不需要训练的 one-hot 基线矩阵。
def cosine(left, right):  # 定义带数值保护的余弦相似度计算。
    denominator = left.norm() * right.norm()  # 计算两个向量范数的乘积。
    return float(torch.dot(left, right) / denominator.clamp_min(1e-12))  # 返回标量余弦值并避免除零。
comparison_pairs = [("无线", "蓝牙", "相关属性"), ("耳机", "降噪", "相关品类属性"), ("耳机", "保护壳", "弱相关品类")]  # 定义有明确业务含义的相似度评测 pair。
print("token pair          业务关系       one-hot余弦")  # 输出基线结果表标题。
for left, right, relation in comparison_pairs:  # 逐组计算不同 token 的 one-hot 相似度。
    similarity = cosine(one_hot[token_to_id[left]], one_hot[token_to_id[right]])  # 查表并计算当前 pair 余弦值。
    print(f"{left + ' / ' + right:<18} {relation:<12} {similarity:>11.3f}")  # 输出 pair、业务解释和基线分数。

token pair          业务关系       one-hot余弦
无线 / 蓝牙            相关属性               0.000
耳机 / 降噪            相关品类属性             0.000
耳机 / 保护壳           弱相关品类              0.000


## 核心实现一：从会话构造加权 Skip-gram 训练样本

窗口大小为 2，窗口内每个有向 `(center, context)` pair 都按会话出现次数累加。负采样分布使用 token 频次的 `0.75` 次方，降低超高频 token 对噪声样本的垄断。下面直接展示频次最高的正 pair 和负采样概率。

In [3]:
pair_weights = Counter()  # 创建有向中心词与上下文词的加权 pair 计数器。
token_frequencies = Counter()  # 创建普通 token 的业务频次计数器。
window_size = 2  # 使用左右各两个位置作为教学版上下文窗口。
for tokens, session_count in sessions:  # 遍历每种会话模式及其出现次数。
    for token in tokens:  # 统计当前会话中的每个普通 token。
        token_frequencies[token] += session_count  # 按会话次数累计 token 频次。
    for center_index, center in enumerate(tokens):  # 枚举每个中心词位置。
        left_edge = max(0, center_index - window_size)  # 计算不会越界的左窗口边界。
        right_edge = min(len(tokens), center_index + window_size + 1)  # 计算不会越界的右窗口边界。
        for context_index in range(left_edge, right_edge):  # 枚举当前中心词窗口内的位置。
            if context_index != center_index:  # 中心词本身不能作为自己的正上下文。
                pair_weights[(center, tokens[context_index])] += session_count  # 按会话次数累加有向正 pair 权重。
center_ids = torch.tensor([token_to_id[center] for center, _ in pair_weights], dtype=torch.long)  # 把全部中心词转换为整数 id 张量。
context_ids = torch.tensor([token_to_id[context] for _, context in pair_weights], dtype=torch.long)  # 把全部正上下文转换为整数 id 张量。
positive_weights = torch.tensor(list(pair_weights.values()), dtype=torch.float32)  # 把业务 pair 次数转换为训练权重。
negative_weights = torch.tensor([0.0] + [float(token_frequencies[token]) ** 0.75 for token in all_words])  # 按 0.75 次方构造含 PAD 的噪声权重。
negative_distribution = negative_weights / negative_weights.sum()  # 归一化得到合法负采样概率分布。
print("加权频次最高的正样本 pair：")  # 输出正训练信号的可读标题。
for (center, context), count in pair_weights.most_common(10):  # 展示最高频的十个有向上下文 pair。
    print(f"  {center:>4} → {context:<4}  权重={count}")  # 输出中心词、上下文词和真实聚合权重。
print("负采样概率最高的 token：", [(token, round(float(negative_distribution[token_to_id[token]]), 3)) for token, _ in token_frequencies.most_common(6)])  # 输出噪声分布中最重要的 token。

加权频次最高的正样本 pair：
    蓝牙 → 耳机    权重=220
    耳机 → 蓝牙    权重=220
    耳机 → 降噪    权重=220
    降噪 → 耳机    权重=220
    无线 → 耳机    权重=200
    耳机 → 无线    权重=200
    音箱 → 便携    权重=135
    便携 → 音箱    权重=135
    无线 → 蓝牙    权重=120
    蓝牙 → 无线    权重=120
负采样概率最高的 token： [('耳机', 0.102), ('蓝牙', 0.086), ('无线', 0.079), ('降噪', 0.069), ('手机', 0.053), ('音箱', 0.048)]


## 核心实现二：手写参数、forward、backward 与更新

模型没有调用现成 `Embedding` 或训练器：两个 `nn.Parameter` 就是中心词表和上下文词表。`forward` 明确计算正样本点积、多个负样本点积和 `softplus` 形式的稳定负采样损失；训练循环真实执行反向传播和手动 SGD。

In [4]:
class SkipGramNegativeSampling(nn.Module):  # 定义最小但完整的 Skip-gram 负采样网络。
    def __init__(self, vocabulary_size, embedding_dim):  # 初始化两个方向的可学习 embedding 表。
        super().__init__()  # 注册基础模块状态以支持参数追踪。
        self.input_weight = nn.Parameter(torch.randn(vocabulary_size, embedding_dim) * 0.08)  # 创建中心词向量参数矩阵。
        self.output_weight = nn.Parameter(torch.randn(vocabulary_size, embedding_dim) * 0.08)  # 创建上下文词向量参数矩阵。
    def forward(self, centers, contexts, negatives):  # 计算每个正 pair 及其负样本的逐样本损失。
        center_vectors = self.input_weight[centers]  # 按中心词 id 直接索引参数矩阵。
        context_vectors = self.output_weight[contexts]  # 按正上下文 id 索引输出参数矩阵。
        negative_vectors = self.output_weight[negatives]  # 按负样本 id 索引三维向量张量。
        positive_logits = (center_vectors * context_vectors).sum(dim=1)  # 计算每个中心词与真实上下文的点积。
        negative_logits = (center_vectors.unsqueeze(1) * negative_vectors).sum(dim=2)  # 计算每个中心词与多个噪声词的点积。
        positive_loss = F.softplus(-positive_logits)  # 用负 log-sigmoid 拉高真实上下文点积。
        negative_loss = F.softplus(negative_logits).sum(dim=1)  # 用负 log-sigmoid 拉低所有噪声点积。
        return positive_loss + negative_loss  # 返回尚未按业务频次加权的逐 pair 损失。
torch.manual_seed(7)  # 固定参数初始化和负采样随机序列以保证输出可复现。
model = SkipGramNegativeSampling(len(vocabulary), 10)  # 创建十维教学 embedding 模型。
learning_rate = 0.08  # 设置适合全批量加权训练的手动 SGD 学习率。
negative_count = 5  # 为每个正 pair 采样五个噪声上下文。
loss_history = []  # 保存关键训练轮次的损失供趋势观察。
generator = torch.Generator().manual_seed(23)  # 创建独立随机生成器固定负采样序列。
for step in range(501):  # 执行真实的前向、反向和参数更新。
    sampled = torch.multinomial(negative_distribution, len(pair_weights) * negative_count, replacement=True, generator=generator)  # 按 0.75 次方分布抽取本轮负样本 id。
    negative_ids = sampled.reshape(len(pair_weights), negative_count)  # 把扁平样本整理为每个正 pair 对应五个负词。
    per_pair_loss = model(center_ids, context_ids, negative_ids)  # 调用自定义 forward 得到逐 pair 负采样损失。
    loss = (per_pair_loss * positive_weights).sum() / positive_weights.sum()  # 按真实会话频次计算加权平均损失。
    loss.backward()  # 通过两个参数矩阵执行真实反向传播。
    with torch.no_grad():  # 关闭更新步骤的梯度记录以避免污染计算图。
        for parameter in model.parameters():  # 逐个更新中心词表和上下文词表。
            parameter -= learning_rate * parameter.grad  # 按当前梯度执行一阶 SGD 更新。
            parameter.grad.zero_()  # 清空梯度避免下一轮错误累积。
    if step in {0, 1, 5, 20, 100, 250, 500}:  # 只记录能说明收敛趋势的代表轮次。
        loss_history.append((step, float(loss.detach())))  # 保存脱离计算图的数值损失。
print("训练轮次 | 加权负采样损失")  # 输出训练轨迹表标题。
for step, loss_value in loss_history:  # 遍历关键轮次展示真实优化过程。
    print(f"{step:>8} | {loss_value:>16.4f}")  # 输出轮次与对应损失。

训练轮次 | 加权负采样损失
       0 |           4.1600
       1 |           4.1604
       5 |           4.1614
      20 |           4.1584
     100 |           4.1356
     250 |           3.8864
     500 |           2.9067


## 结果表：从 one-hot 正交到可比较的稠密几何

评估使用中心与上下文两张表的平均值，并做 L2 归一化。这里不声称小语料得到通用语义，只检查训练目标是否让同一浏览场景中的 pair 比无关品类更接近，并打印最近邻供人工审计。

In [5]:
with torch.no_grad():  # 评估阶段不需要构建梯度图。
    learned_vectors = (model.input_weight + model.output_weight) / 2.0  # 合并中心词与上下文词视角得到单一词向量表。
    learned_vectors = F.normalize(learned_vectors, dim=1)  # 对每一行执行 L2 归一化便于直接点积比较。
def nearest_neighbors(token, count=4):  # 定义最近邻检查函数以发现训练后几何结构。
    token_id = token_to_id[token]  # 查找目标 token 的稳定整数编号。
    scores = learned_vectors @ learned_vectors[token_id]  # 计算目标向量与完整词表的余弦相似度。
    ranked_ids = torch.argsort(scores, descending=True).tolist()  # 按相似度从高到低取得候选编号。
    neighbors = [(vocabulary[index], float(scores[index])) for index in ranked_ids if vocabulary[index] not in {token, "[PAD]"}]  # 排除目标自身和 padding。
    return neighbors[:count]  # 返回指定数量的最高相似邻居。
print("token pair          one-hot余弦  learned余弦  业务关系")  # 输出训练前后相似度对照表标题。
learned_pair_scores = {}  # 保存业务 pair 的学习后分数供测试使用。
for left, right, relation in comparison_pairs:  # 在与基线完全相同的 pair 上评估学习结果。
    baseline_score = cosine(one_hot[token_to_id[left]], one_hot[token_to_id[right]])  # 重新取得 one-hot 基线分数。
    learned_score = float(torch.dot(learned_vectors[token_to_id[left]], learned_vectors[token_to_id[right]]))  # 计算学习后归一化向量点积。
    learned_pair_scores[(left, right)] = learned_score  # 保存当前业务 pair 的稠密相似度。
    print(f"{left + ' / ' + right:<18} {baseline_score:>11.3f} {learned_score:>12.3f}  {relation}")  # 输出同一 pair 的训练前后变化。
for token in ["无线", "蓝牙", "耳机", "保护壳"]:  # 为四个代表 token 展示可人工检查的局部邻域。
    print(f"{token} 的最近邻：", [(neighbor, round(score, 3)) for neighbor, score in nearest_neighbors(token)])  # 输出邻居名称和真实余弦值。

token pair          one-hot余弦  learned余弦  业务关系
无线 / 蓝牙                  0.000        0.485  相关属性
耳机 / 降噪                  0.000        0.594  相关品类属性
耳机 / 保护壳                 0.000       -0.257  弱相关品类
无线 的最近邻： [('耳机', 0.648), ('蓝牙', 0.485), ('运动', 0.4), ('通话', 0.356)]
蓝牙 的最近邻： [('通话', 0.87), ('耳机', 0.83), ('无线', 0.485), ('降噪', 0.416)]
耳机 的最近邻： [('蓝牙', 0.83), ('通话', 0.731), ('无线', 0.648), ('降噪', 0.594)]
保护壳 的最近邻： [('防摔', 0.577), ('户外', 0.348), ('透明', 0.343), ('手机', 0.326)]


## 结果解读

one-hot 对所有不同 token 都返回零，无法表达任何软相似。经过真实 `forward/backward` 后，“无线/蓝牙”和“耳机/降噪”的分数由共同浏览上下文决定，而“耳机/保护壳”受不同会话簇约束。最近邻是必要的定性审计：如果只看 loss，就发现不了模型把高频词全部挤成一团、把敏感属性编码进几何结构等问题。

## 失败案例：mean pooling 把 PAD 当成正文

同一句“蓝牙 耳机”在不同 batch 中可能被补到不同长度。如果直接对全部位置求均值，一个未冻结或发生漂移的 PAD 向量会改变句向量；修复是用 attention mask 做加权和，并除以真实 token 数。

In [6]:
sentence_ids = torch.tensor([token_to_id["蓝牙"], token_to_id["耳机"]], dtype=torch.long)  # 创建未补齐的真实句子 token id。
padded_ids = torch.tensor([token_to_id["蓝牙"], token_to_id["耳机"], token_to_id["[PAD]"], token_to_id["[PAD]"]], dtype=torch.long)  # 创建同一句话的四长度 batch 版本。
pooling_table = learned_vectors.clone()  # 复制学习后向量以单独复现 padding 漂移问题。
pooling_table[token_to_id["[PAD]"]] = torch.linspace(-1.5, 1.5, pooling_table.shape[1])  # 模拟训练中未屏蔽 PAD 后产生的非零向量。
plain_vector = pooling_table[sentence_ids].mean(dim=0)  # 对未补齐句子直接求均值得到参考向量。
broken_vector = pooling_table[padded_ids].mean(dim=0)  # 错误地把两个 PAD 位置一起纳入均值。
attention_mask = torch.tensor([1.0, 1.0, 0.0, 0.0])  # 为真实 token 标一、padding 位置标零。
masked_sum = (pooling_table[padded_ids] * attention_mask.unsqueeze(1)).sum(dim=0)  # 用 mask 排除 PAD 后计算真实 token 向量和。
fixed_vector = masked_sum / attention_mask.sum().clamp_min(1.0)  # 除以真实 token 数得到稳定的 masked mean。
broken_difference = float((plain_vector - broken_vector).abs().max())  # 量化错误 pooling 对同一句话造成的最大维度漂移。
fixed_difference = float((plain_vector - fixed_vector).abs().max())  # 量化 mask 修复后的剩余差异。
print("未补齐句向量前四维：", [round(float(value), 4) for value in plain_vector[:4]])  # 输出参考句向量的可读切片。
print("错误 mean 前四维：", [round(float(value), 4) for value in broken_vector[:4]], f"，最大漂移={broken_difference:.4f}")  # 展示 padding 污染后的向量变化。
print("masked mean 前四维：", [round(float(value), 4) for value in fixed_vector[:4]], f"，最大漂移={fixed_difference:.4f}")  # 展示正确 mask 后完全一致的结果。

未补齐句向量前四维： [-0.0121, -0.4589, -0.4318, 0.1517]
错误 mean 前四维： [-0.7561, -0.8128, -0.6326, -0.1742] ，最大漂移=0.9144
masked mean 前四维： [-0.0121, -0.4589, -0.4318, 0.1517] ，最大漂移=0.0000


## 生产差距与落地清单

教学数据只有九种会话，负样本也可能误抽到真实相关词；线上应做大规模流式样本、去偏与假负例控制，并按时间切分验证。训练与服务必须固定 tokenizer、token id、向量维度、归一化和 pooling 版本；监控 OOV/byte fallback、向量范数、热门词近邻、分语言召回率和漂移。若用于 ANN 检索，还要共同验证量化误差与索引版本，而不是只离线看余弦。

## 最小回归测试

断言只保护梯度确实更新、相关性排序和 mask 不变量；上面的 pair 构造、loss 曲线、近邻与失败对照才是完整证据。

In [7]:
assert loss_history[-1][1] < loss_history[0][1]  # 验证真实反向传播让加权负采样损失下降。
assert learned_pair_scores[("无线", "蓝牙")] > learned_pair_scores[("耳机", "保护壳")]  # 验证相关属性 pair 比跨品类 pair 更接近。
assert learned_pair_scores[("耳机", "降噪")] > learned_pair_scores[("耳机", "保护壳")]  # 验证同会话品类属性超过无关配件关系。
assert broken_difference > 0.1  # 固化未屏蔽 PAD 会显著改变句向量的失败案例。
assert fixed_difference < 1e-7  # 验证 masked mean 与未补齐句子的向量完全一致。
assert token_to_id["[PAD]"] == 0  # 验证特殊 token 编号合同保持稳定。
print("最小回归测试通过：训练更新、语义 pair 排序和 padding mask 均符合预期。")  # 输出完整执行成功的明确结论。

最小回归测试通过：训练更新、语义 pair 排序和 padding mask 均符合预期。
